# Particle Genealogy and Degeneracy

## Historical problem

Particle filters can approximate filtering distributions well in forward time, but repeated resampling creates a less obvious problem: **genealogical degeneracy**. After many resampling steps, a large final particle population may descend from only a few ancient ancestors.

This matters because filtering asks for the current hidden state, while smoothing asks for earlier hidden states conditional on all later data. Path degeneracy makes smoothing much harder.

This notebook traces particle ancestry directly.

In [ ]:
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().resolve().parents[0]
SHARED = ROOT / "00_shared"
if str(SHARED) not in sys.path:
    sys.path.append(str(SHARED))

from plotting import save_fig, set_plot_style

set_plot_style()
rng = np.random.default_rng(321)

## Simulate a nonlinear hidden-state model

We use a one-dimensional latent process with nonlinear observations:

$$
x_t = 0.9 x_{t-1} + \varepsilon_t, \qquad y_t = \frac{x_t^2}{20} + \eta_t.
$$

This is enough to produce meaningful reweighting and repeated resampling.

In [ ]:
T = 45
N = 350
state_sd = 1.0
obs_sd = 0.7

x_true = np.zeros(T)
x_true[0] = rng.normal(0.0, 1.0)
for t in range(1, T):
    x_true[t] = 0.9 * x_true[t - 1] + rng.normal(0.0, state_sd)

y_obs = x_true**2 / 20.0 + rng.normal(0.0, obs_sd, size=T)

print("Latent trajectory and observations simulated.")

## Particle filter with ancestry tracking

In [ ]:
def systematic_resample(weights, rng):
    n = len(weights)
    positions = (rng.random() + np.arange(n)) / n
    cumulative = np.cumsum(weights)
    indexes = np.zeros(n, dtype=int)
    i = 0
    j = 0
    while i < n:
        if positions[i] < cumulative[j]:
            indexes[i] = j
            i += 1
        else:
            j += 1
    return indexes


particles = rng.normal(0.0, 2.0, size=N)
weights = np.full(N, 1.0 / N)
ancestors = np.zeros((T, N), dtype=int)
particle_history = np.zeros((T, N))
filtered_means = np.zeros(T)
ess = np.zeros(T)

for t in range(T):
    if t > 0:
        particles = 0.9 * particles + rng.normal(0.0, state_sd, size=N)

    logw = -0.5 * ((y_obs[t] - particles**2 / 20.0) / obs_sd) ** 2
    logw = logw - np.max(logw)
    w = np.exp(logw)
    weights = w / np.sum(w)
    filtered_means[t] = np.sum(weights * particles)
    ess[t] = 1.0 / np.sum(weights**2)
    particle_history[t] = particles

    if t == 0:
        ancestors[t] = np.arange(N)
    else:
        if ess[t] < N / 2:
            idx = systematic_resample(weights, rng)
            ancestors[t] = idx
            particles = particles[idx]
            weights = np.full(N, 1.0 / N)
            particle_history[t] = particles
        else:
            ancestors[t] = np.arange(N)


def trace_unique_ancestor_counts(ancestors):
    current = np.arange(ancestors.shape[1])
    counts = [len(np.unique(current))]
    for t in range(ancestors.shape[0] - 1, 0, -1):
        current = ancestors[t][current]
        counts.append(len(np.unique(current)))
    return np.array(counts)


unique_counts_backwards = trace_unique_ancestor_counts(ancestors)
lags = np.arange(len(unique_counts_backwards))

subset = rng.choice(N, size=30, replace=False)
lineages = np.zeros((T, len(subset)), dtype=int)
lineages[-1] = subset
for k in range(len(subset)):
    idx = subset[k]
    for t in range(T - 1, 0, -1):
        idx = ancestors[t][idx]
        lineages[t - 1, k] = idx

print("Ancestry tracing complete.")

## Degeneracy diagnostics

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 11))

axes[0].plot(x_true, color="#111111", lw=2, label="True state")
axes[0].plot(filtered_means, color="#d62728", lw=2, label="Filtered mean")
axes[0].set_title("Filtering still works well for the current state")
axes[0].set_xlabel("Time")
axes[0].set_ylabel("State")
axes[0].legend()

axes[1].plot(ess, color="#54a24b", lw=2)
axes[1].axhline(N / 2, color="#d62728", ls="--", label="Resampling threshold")
axes[1].set_title("Effective sample size over time")
axes[1].set_xlabel("Time")
axes[1].set_ylabel("ESS")
axes[1].legend()

axes[2].plot(lags, unique_counts_backwards, color="#4c78a8", lw=2)
axes[2].set_title("How many distinct ancestors survive as we look backwards?")
axes[2].set_xlabel("Lag backwards from final time")
axes[2].set_ylabel("Unique ancestors")

fig.tight_layout()
save_fig(fig, Path("figs") / "ancestor_collapse_and_ess.png")
plt.show()

fig, ax = plt.subplots(figsize=(9, 6))
times = np.arange(T)
for k in range(lineages.shape[1]):
    ax.plot(times, lineages[:, k], alpha=0.55, lw=1)
ax.set_title("Genealogy of a subset of final particles")
ax.set_xlabel("Time")
ax.set_ylabel("Ancestor index")
fig.tight_layout()
save_fig(fig, Path("figs") / "particle_genealogy.png")
plt.show()

## Interpretation

The key lesson is asymmetric:

- **Filtering** can remain accurate because the current weighted particles still represent the current state well.
- **Smoothing** becomes harder because the ancestral diversity of those particles collapses as we look backward in time.

So resampling solves one problem, weight degeneracy, while gradually creating another, path degeneracy.

## References

- Doucet, de Freitas, and Gordon (2001), *Sequential Monte Carlo Methods in Practice*.
- Later SMC literature on genealogical degeneracy and smoothing.